# Module 5.2: The Training Loop

Once we have our Cross-Entropy Loss (which tells us how wrong the model is), we need to actually update the millions of parameters inside the Transformer to be *less* wrong.

In this notebook, we look at the magic of **Backpropagation** and **Optimizers**, the engine that drives all Neural Network training.

## 1. Backpropagation (Calculus in Disguise)

PyTorch has an engine called `Autograd` that mathematically remembers every matrix multiplication, addition, and scaling function our data went through during the Forward Pass.

When we call `loss.backward()`, PyTorch walks backward across the entire graph and calculates the **Gradient** (the slope) for every single weight in our Transformer. A gradient tells us: *"If I increase this weight by a tiny amount, will the loss go up or down, and by how much?"* 

## 2. The Optimizer (AdamW)

Subtracting the raw gradients from our weights is called basic Stochastic Gradient Descent (SGD). However, for deep LLMs, this causes training to be unstable.

Modern LLMs use **AdamW** (Adaptive Moment Estimation with Weight Decay):
1. **Adaptive:** It keeps track of the 'momentum' of gradients, so if we constantly push a weight in one direction, it accelerates it.
2. **Weight Decay:** It constantly tries to push weights towards `0.0` slightly. This prevents any single weight from becoming too huge and breaking the model (regularization).

Let's build a mini training loop!

In [ ]:
import torch
import torch.nn as nn

# 1. A tiny mock model (Imagine this is our 96-layer Transformer!)
model = nn.Sequential(
    nn.Linear(256, 512),
    nn.ReLU(),
    nn.Linear(512, 1000) # Outputting 1000 raw logits for 1000 vocab words
)

# 2. The Optimizer (AdamW)
# We hand it the actual memory addresses of our model's weights so it can edit them!
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

# 3. A dummy dataset 
x_inputs = torch.randn(32, 256)   # Batch of 32 contexts
y_targets = torch.randint(0, 1000, (32,)) # 32 target words to predict

# --- THE SACRED TRAINING LOOP ---
epochs = 5
for epoch in range(epochs):
    
    # Step A: Clear old gradients (don't let them accumulate from last step!)
    optimizer.zero_grad()
    
    # Step B: Forward Pass (Predict)
    logits = model(x_inputs)
    
    # Step C: Calculate Loss (How wrong were we?)
    loss = nn.functional.cross_entropy(logits, y_targets)
    
    # Step D: Backward Pass (Calculate the slopes using Calculus)
    loss.backward()
    
    # Step E: Optimize! (Physically edit the weights)
    optimizer.step()
    
    print(f"Epoch {epoch+1}/5 | Loss: {loss.item():.4f}")

print("Training complete! Notice how the Loss magically decreases as the weights 'learn'.")

## Summary

This small 5-step loop is the engine that trains everything from a small digit classifier (MNIST) up to a 400 Billion parameter Llama 3 model running on a cluster of 16,000 GPUs.

Once the loss hits its floor across trillions of web tokens, the result is a **Foundation Model** (or Base Model). It knows grammar, facts, and logic, but it doesn't know how to chat. If you say "Hello", it might complete it with "Hello, World!" instead of "Hi, how can I help you today?". 

To turn it into an assistant, we need **Module 6: Fine-Tuning and Alignment**!